# Versioned external evidence
This independent, fabricated M1 → M2 → M3 example keeps SCV and VCV claims separate. A link is neither carriage nor a clinical conclusion.

In [ ]:
import os, sys

in_colab = "google.colab" in sys.modules
PROFILE = os.environ.get(
    "GENOME_EVIDENCE_PROFILE", "personal_drive" if in_colab else "synthetic_ci"
)
REPOSITORY_URL = "https://github.com/jcollins-bioinfo/genome-evidence.git"
REPOSITORY_REF = os.environ.get("GENOME_EVIDENCE_GIT_REF", "main")
WORKSPACE_ROOT = os.environ.get(
    "GENOME_EVIDENCE_WORKSPACE", "/content/drive/MyDrive/genome-evidence-private"
)
SUBJECT_ID = os.environ.get("GENOME_EVIDENCE_SUBJECT_ID", "subject-0001")

In [ ]:
import importlib
import importlib.metadata
import json
import subprocess
from hashlib import sha256
from pathlib import Path

if PROFILE not in {"personal_drive", "synthetic_ci"}:
    raise ValueError("PROFILE must be personal_drive or synthetic_ci")

CHECKOUT = Path("/content/genome-evidence-src")
if PROFILE == "personal_drive":
    if "google.colab" in sys.modules:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
    if CHECKOUT.exists():
        remote = subprocess.run(
            ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
            timeout=30,
        ).stdout.strip()
        if remote != REPOSITORY_URL:
            raise RuntimeError("Unexpected checkout remote; move the checkout aside and rerun")
        dirty = subprocess.run(
            ["git", "-C", str(CHECKOUT), "status", "--porcelain"],
            check=True,
            capture_output=True,
            text=True,
            timeout=30,
        ).stdout
        if dirty:
            raise RuntimeError("Checkout is dirty; preserve or move it aside and rerun")
    else:
        subprocess.run(
            ["git", "clone", "--no-checkout", REPOSITORY_URL, str(CHECKOUT)],
            check=True,
            timeout=180,
        )
    subprocess.run(
        ["git", "-C", str(CHECKOUT), "fetch", "--force", "origin", REPOSITORY_REF],
        check=True,
        timeout=180,
    )
    RESOLVED_COMMIT = subprocess.run(
        ["git", "-C", str(CHECKOUT), "rev-parse", "--verify", "FETCH_HEAD^{commit}"],
        check=True,
        capture_output=True,
        text=True,
        timeout=30,
    ).stdout.strip()
    loaded_package_modules = [
        module
        for name, module in sys.modules.items()
        if name == "genome_evidence" or name.startswith("genome_evidence.")
    ]
    if loaded_package_modules:
        current_commit = subprocess.run(
            ["git", "-C", str(CHECKOUT), "rev-parse", "HEAD^{commit}"],
            check=True,
            capture_output=True,
            text=True,
            timeout=30,
        ).stdout.strip()
        loaded_paths = [
            Path(str(module_file)).resolve()
            for module in loaded_package_modules
            if (module_file := getattr(module, "__file__", None)) is not None
        ]
        if current_commit != RESOLVED_COMMIT or any(
            not path.is_relative_to(CHECKOUT.resolve()) for path in loaded_paths
        ):
            raise RuntimeError(
                "genome_evidence modules from another revision are already loaded; "
                "restart the runtime and rerun from the first cell"
            )
    subprocess.run(
        ["git", "-C", str(CHECKOUT), "checkout", "--detach", RESOLVED_COMMIT],
        check=True,
        timeout=60,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "-e",
            f"{CHECKOUT}[notebook]",
        ],
        check=True,
        timeout=600,
    )
    source_root = (CHECKOUT / "src").resolve()
    package_init = source_root / "genome_evidence" / "__init__.py"
    if not package_init.is_file():
        raise RuntimeError("Resolved checkout does not contain the genome_evidence package")
    source_path = str(source_root)
    if source_path not in sys.path:
        sys.path.insert(0, source_path)
    importlib.invalidate_caches()
else:
    RESOLVED_COMMIT = "installed-ci-package"

genome_evidence = importlib.import_module("genome_evidence")
PACKAGE_ORIGIN = Path(genome_evidence.__file__).resolve()
if PROFILE == "personal_drive" and not PACKAGE_ORIGIN.is_relative_to(CHECKOUT.resolve()):
    raise RuntimeError("genome_evidence import origin is outside the resolved checkout")
INSTALLED_VERSION = importlib.metadata.version("genome-evidence")
LOCK_SHA256 = (
    sha256((CHECKOUT / "uv.lock").read_bytes()).hexdigest() if PROFILE == "personal_drive" else None
)
SANITIZED_IMPORT_PATH = (
    str(PACKAGE_ORIGIN.relative_to(CHECKOUT))
    if PROFILE == "personal_drive"
    else "installed-ci-package"
)
BOOTSTRAP_STATUS = {
    "profile": PROFILE,
    "requested_ref": REPOSITORY_REF,
    "resolved_commit": RESOLVED_COMMIT,
    "version": INSTALLED_VERSION,
    "import_path": SANITIZED_IMPORT_PATH,
    "lock_sha256": LOCK_SHA256,
    "lock_equivalent": PROFILE != "personal_drive",
}
print(json.dumps(BOOTSTRAP_STATUS, sort_keys=True))

In [ ]:
from genome_evidence.workspace import validate_workspace

if PROFILE == "personal_drive":
    workspace = validate_workspace(Path(WORKSPACE_ROOT))
else:
    assert PROFILE == "synthetic_ci"
    workspace = None

In [ ]:
# ruff: noqa
import json, tempfile
from pathlib import Path
from genome_evidence.ingest import Ingest23andMeConfig, ingest_23andme
from genome_evidence.normalization import NormalizationConfig, normalize_m1_run
from genome_evidence.evidence import ingest_clinvar_vcv, link_external_evidence

root = Path(tempfile.mkdtemp())
source = root / "source.txt"
source.write_text("# genome build: GRCh38\nsynthetic_marker\t1\t5\tAA\n")
markers = root / "markers.json"
markers.write_text(
    json.dumps(
        [
            {
                "marker_id": "synthetic_marker",
                "assembly": "GRCh38",
                "chromosome": "1",
                "position": 5,
                "reference": "A",
                "alternate": "G",
                "orientation": "none",
                "orientation_authoritative": True,
            }
        ]
    )
)
fasta = root / "ref.fa"
fasta.write_text(">1\n" + "A" * 20 + "\n")
ingest_23andme(source, root / "m1", Ingest23andMeConfig(genome_build_override="GRCh38"))
normalize_m1_run(
    root / "m1",
    root / "m2",
    NormalizationConfig(marker_definitions=markers, target_reference=fasta),
)
xml = root / "synthetic.xml"
xml.write_text(
    """<ReleaseSet Dated="2026-07-01" ReleaseID="synthetic-release"><VariationArchive Accession="VCV999999001" Version="1"><ClassifiedRecord><SimpleAllele AlleleID="1"><SequenceLocation Assembly="GRCh38" Chr="1" positionVCF="5" referenceAlleleVCF="A" alternateAlleleVCF="G"/></SimpleAllele><Classifications><GermlineClassification><Description>synthetic aggregate term</Description></GermlineClassification></Classifications><ClinicalAssertion Accession="SCV999999001" Version="1"><Submitter Name="Fabricated Submitter"/><GermlineClassification><Description>synthetic submitted term A</Description></GermlineClassification></ClinicalAssertion><ClinicalAssertion Accession="SCV999999002" Version="1"><GermlineClassification><Description>synthetic submitted term B</Description></GermlineClassification></ClinicalAssertion></ClassifiedRecord></VariationArchive><VariationArchive Accession="VCV999999002" Version="1"><ClassifiedRecord><Haplotype/><Classifications><OncogenicityClassification><Description>synthetic unsupported term</Description></OncogenicityClassification></Classifications></ClassifiedRecord></VariationArchive></ReleaseSet>"""
)
evidence = ingest_clinvar_vcv(xml, root / "evidence")
annotation = link_external_evidence(root / "m2", root / "evidence", root / "annotation")
assert len([a for a in evidence.assertions if a.scv_accession]) == 2
assert len([a for a in evidence.assertions if not a.scv_accession]) == 2
assert {x.outcome.value for x in annotation.links} == {"matched", "unsupported"}
assert all(not hasattr(x, "genotype") for x in annotation.links)
assert "does not establish" in (root / "annotation/annotation_report.md").read_text()
[(a.logical_source_key, a.source_classification_terms) for a in evidence.assertions]